# unialg Explorer

A gentle tour of the current `unialg` API.

A **morphism** is a typed arrow. You can run it, compose it, branch it, add parameters, add effects, and lift it through reusable data shapes.

The notebook keeps backend setup in `explore_support.py` so the main path can stay focused on the algebraic API.

## Setup

Run this once before the examples. It loads the local package and imports a small set of demo morphisms.

In [1]:
import sys
_VENV = "/home/scanbot/unialg/.venv/lib/python3.12/site-packages"
_SRC = "/home/scanbot/unialg/src"
for _p in (_VENV, _SRC):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from unialg.main import lower as _lower_node, run
from unialg.objects import LIST, MAYBE, ProductType, SumType
from unialg.syntax import expressions as expr
from unialg.syntax.expressions import Id, One, Const, Sum, Prod
from unialg.semantics.functors import Functor, apply_poly, poly_fmap
from unialg.semantics.morphisms import (
    Morphism,
    _assoc as assoc,
    _copy as copy,
    _delete as delete,
    _fst as fst,
    _inl as inl,
    _inr as inr,
    _snd as snd,
    _symmetry as symmetry,
    case,
    compose,
    identity,
    pair,
    par,
)
from unialg.semantics.optics import Optic, ana, cata, hylo
from unialg.tensors.semirings import Semiring


def lower(morphism, graph=None):
    return _lower_node(morphism.node, graph)


def act(optic, h):
    return optic.act(h)


def act_forward(optic, h):
    return optic.act_forward(h)


def act_backward(optic, h):
    return optic.act_backward(h)


def compose_optic(outer, inner):
    return outer.compose(inner)

from explore_support import *

print("Ready.")

Ready.


## 1. Plain Morphisms

`add1` and `double` are ordinary morphisms from `Int` to `Int`. The helper `show_morphism` prints the logical type, and `run_int` runs a morphism on an integer input.

In [2]:
print("add1: ", show_morphism(add1))
print("double:", show_morphism(double))

print("add1(5)  =", run_int(add1, 5))
print("double(5) =", run_int(double, 5))

add1:  Int -> Int
double: Int -> Int
add1(5)  = 6
double(5) = 10


## 2. Composition

`compose(f, g)` means: run `f`, then run `g`. The output type of `f` must match the input type of `g`.

In [3]:
add_then_double = compose(add1, double)

print("add_then_double:", show_morphism(add_then_double))
print("add_then_double(3) =", run_int(add_then_double, 3))

add_then_double: Int -> Int
add_then_double(3) = 8


In [4]:
try:
    compose(add1, add_pair)
except Exception as e:
    print(type(e).__name__ + ": add1 returns Int, but add_pair expects Int x Int")

MorphismError: add1 returns Int, but add_pair expects Int x Int


## 3. Products

Products let us talk about pairs.

`copy` duplicates one input. `fst` and `snd` project from a pair. `pair(f, g)` sends the same input to both morphisms. `par(f, g)` sends the left input to `f` and the right input to `g`.

In [5]:
copy_int = copy(INT)
fst_int = fst(INT_PAIR)
snd_int = snd(INT_PAIR)

print("copy:", show_morphism(copy_int), "   copy(7) =", run_pair(copy_int, 7))
print("fst: ", show_morphism(fst_int), "   fst(3, 9) =", run_int_from_pair(fst_int, (3, 9)))
print("snd: ", show_morphism(snd_int), "   snd(3, 9) =", run_int_from_pair(snd_int, (3, 9)))

copy: Int -> Int x Int    copy(7) = (7, 7)
fst:  Int x Int -> Int    fst(3, 9) = 3
snd:  Int x Int -> Int    snd(3, 9) = 9


In [6]:
same_input = pair(add1, double)
side_by_side = par(add1, double)

print("pair(add1, double):", show_morphism(same_input))
print("pair(add1, double)(5) =", run_pair(same_input, 5))

print("par(add1, double): ", show_morphism(side_by_side))
print("par(add1, double)(5, 6) =", run_pair(side_by_side, (5, 6)))

pair(add1, double): Int -> Int x Int
pair(add1, double)(5) = (6, 10)
par(add1, double):  Int x Int -> Int x Int
par(add1, double)(5, 6) = (6, 12)


## 4. Sums

Sums represent a choice: a value arrives from the left side or the right side.

`case(f, g)` branches on that choice. Left values go to `f`; right values go to `g`.

In [7]:
branch = case(add1, double)

print("branch:", show_morphism(branch))
print("branch(Left 5)  =", run_left_int(branch, 5))
print("branch(Right 5) =", run_right_int(branch, 5))

branch: Int + Int -> Int
branch(Left 5)  = 6
branch(Right 5) = 10


## 5. Parameters

A parametric morphism has an extra environment value. In this example, the parameter is an integer that gets added to the input.

The important surface idea is simple: provide a parameter and an input. The support helper owns the runtime packing.

In [8]:
print("add_param:", show_morphism(add_param))
print("add_param(param=10, value=3) =", run_para_int(add_param, param=10, value=3))

add_param: param Int, input Int -> Int
add_param(param=10, value=3) = 13


In [9]:
two_params = compose(add_param, add_param_again)

print("compose(add_param, add_param_again):", show_morphism(two_params))
print("with f_param=10, g_param=20, value=3:",
      run_composed_para_int(two_params, f_param=10, g_param=20, value=3))

compose(add_param, add_param_again): param Int x Int, input Int -> Int
with f_param=10, g_param=20, value=3: 33


## 6. Effects

A lax morphism returns its result inside an effect.

Here `Maybe` means the result is wrapped as an optional value, and `List` means the morphism can return multiple results. Composition sequences those effects automatically.

In [10]:
maybe_pipeline = compose(maybe_add1, maybe_double)

print("maybe_pipeline:", show_morphism(maybe_pipeline))
print("maybe_pipeline(3) =", run_maybe_int(maybe_pipeline, 3))

maybe_pipeline: Int -> Maybe Int
maybe_pipeline(3) = 8


In [11]:
list_pipeline = compose(list_step, list_double)

print("list_pipeline:", show_morphism(list_pipeline))
print("list_pipeline(3) =", run_list_int(list_pipeline, 3))

list_pipeline: Int -> List Int
list_pipeline(3) = [6, 8]


## 7. Shapes / Polynomial Functors

A polynomial functor is a reusable data shape. `Id()` marks the places where values of the current type live.

`poly_fmap(shape, h)` lifts a morphism through the shape, applying `h` at every `Id()` position.

In [12]:
pair_shape = Prod(Id(), Id())

print("shape:", expr.pretty(pair_shape))
print("shape applied to Int:", show_type(apply_poly(pair_shape, INT)))

shape: X * X
shape applied to Int: Int x Int


In [13]:
lifted = poly_fmap(Functor("Test", pair_shape), add1)

print("lifted add1:", show_morphism(lifted))
print("lifted add1 on (3, 7) =", run_pair(lifted, (3, 7)))

lifted add1: Int x Int -> Int x Int
lifted add1 on (3, 7) = (4, 8)


In [14]:
lifted_param = poly_fmap(Functor("Test", pair_shape), add_param)

print("lifted add_param:", show_morphism(lifted_param))
print("param=10, value=(3, 7):", run_para_pair(lifted_param, param=10, value=(3, 7)))

lifted add_param: param Int, input Int x Int -> Int x Int
param=10, value=(3, 7): (13, 17)


In [15]:
lifted_maybe = poly_fmap(Functor("Test", pair_shape), maybe_add1)

print("lifted maybe_add1:", show_morphism(lifted_maybe))
print("lifted maybe_add1 on (3, 7) =", run_maybe_pair(lifted_maybe, (3, 7)))

lifted maybe_add1: Int x Int -> Maybe (Int x Int)
lifted maybe_add1 on (3, 7) = (4, 8)


In [16]:
sum_shape = Sum(Id(), Id())
lifted_sum = poly_fmap(Functor("Test", sum_shape), add1)

print("sum shape:", expr.pretty(sum_shape))
print("Left 5  ->", run_sum_int(lifted_sum, side="left", value=5))
print("Right 5 ->", run_sum_int(lifted_sum, side="right", value=5))

sum shape: X + X
Left 5  -> Left 6
Right 5 -> Right 6


## 8. Structural Morphisms — assoc and symmetry

`assoc` and `symmetry` are canonical structural morphisms built from types alone — no primitives needed.

`assoc((A×B)×C)` gives the reassociation `(A×B)×C → A×(B×C)`.  
`symmetry(A×B)` gives the swap `A×B → B×A`.

In [17]:
assoc_m = assoc(INT_TRIPLE_L)
print("assoc:", show_morphism(assoc_m))
print("assoc((1,2),3) =", run_assoc(assoc_m, 1, 2, 3))

sym_m = symmetry(INT_PAIR)
print("symmetry:", show_morphism(sym_m))
print("symmetry(3,7) =", run_pair(sym_m, (3, 7)))

assoc: Int x Int x Int -> Int x Int x Int
assoc((1,2),3) = (1, (2, 3))
symmetry: Int x Int -> Int x Int
symmetry(3,7) = (7, 3)


## 8. Lowering

The examples above use friendly runners. Underneath, `lower(morphism)` compiles the morphism expression into a raw Hydra term. `run(morphism, argument, ctx, graph)` applies that term and reduces it.

You usually do not need to look at the lowered term while using the API, but it is useful when checking the compiler boundary.

In [18]:
compiled = lower(add_then_double, graph)

print("expression:", expr.pretty(add_then_double.node))
print("lowered term class:", type(compiled).__name__)


expression: (prim ; prim)
lowered term class: TermLambda


## 9. What Can Be Composed?

`compose(f, g)` accepts `Morphism` objects. It does not accept raw Python functions directly.

A Python function can still participate, but it first has to become a Hydra `Primitive`, then a `Morphism` can point at that primitive. That backend pattern belongs in fixture/support code, not in the main tutorial path.

In [19]:
print("compose takes Morphism values:")
print("  add1 is", type(add1).__name__)
print("  lambda x: x + 1 is", type(lambda x: x + 1).__name__)

compose takes Morphism values:
  add1 is Morphism
  lambda x: x + 1 is function


## 10. Appendix: Where Did The Fixtures Come From?

The sample morphisms and notebook-friendly runners live in `explore_support.py`.

That file contains the raw Hydra details: primitive names, `expr.Prim(...)` leaves, argument packing, result unwrapping, and the `ctx` / `graph` used by `run`. Keeping that code out of the main path makes this notebook about the `unialg` API rather than Hydra plumbing.

## 11. Try It Yourself

Some small experiments:

- Compose `double` then `add3` and run it on `5`.
- Build `pair(add1, add3)` and run it on `10`.
- Build `par(double, add3)` and run it on `(2, 8)`.
- Lift `double` through `pair_shape` with `poly_fmap`.
- Compose a plain morphism with a Maybe morphism and check the result.

In [20]:
double_then_add3 = compose(double, add3)

print("double_then_add3:", show_morphism(double_then_add3))
print("double_then_add3(5) =", run_int(double_then_add3, 5))

double_then_add3: Int -> Int
double_then_add3(5) = 13


In [21]:
add3_then_double = compose(add3, double)
thendoubleagain = pair(add3_then_double, double)
print(run_pair(thendoubleagain, 21))

(48, 42)


## 12. Optics

An `Optic` is a triple `(functor, forward, backward)` where the functor describes the container shape and the two morphisms decompose and reconstruct around it.

`act(optic, h)` applies `h` through the optic: decompose with `forward`, lift `h` through the functor with `poly_fmap`, then reconstruct with `backward`.

Here a **pair lens** focuses on the first element of `Int × Int` using the functor `F(X) = X × Int`.

In [22]:
pair_lens = Optic(
    functor=Functor("PairFirst", Prod(Id(), Const(INT))),
    forward=identity(INT_PAIR),
    backward=identity(INT_PAIR),
)

print("pair_lens.source:", show_type(pair_lens.source))
print("pair_lens.target:", show_type(pair_lens.target))
print("pair_lens.focus: ", show_type(pair_lens.focus))

lifted = act(pair_lens, add1)
print("act(pair_lens, add1):", show_morphism(lifted))
print("act(pair_lens, add1)(3, 7) =", run_pair(lifted, (3, 7)))

pair_lens.source: Int x Int
pair_lens.target: Int x Int
pair_lens.focus:  Int
act(pair_lens, add1): Int x Int -> Int x Int
act(pair_lens, add1)(3, 7) = (4, 7)


## 13. Semiring

A `Semiring` captures the algebraic structure of two binary operations on a carrier type. All four components (`plus`, `times`, `zero`, `one`) are `Morphism` objects, so they participate in the typed composition machinery.

In [23]:
int_semiring = Semiring(
    name="int-add-mul",
    carrier=INT,
    plus=add_pair,
    times=mul_pair,
    zero=const_zero,
    one=const_one,
)

print("name:    ", int_semiring.name)
print("carrier: ", show_type(int_semiring.carrier))
print("plus:    ", show_morphism(int_semiring.plus))
print("times:   ", show_morphism(int_semiring.times))
print("zero:    ", show_morphism(int_semiring.zero))
print("one:     ", show_morphism(int_semiring.one))

print("plus(3, 4) =", run_int_from_pair(int_semiring.plus, (3, 4)))
print("times(3,4) =", run_int_from_pair(int_semiring.times, (3, 4)))

name:     int-add-mul
carrier:  Int
plus:     Int x Int -> Int
times:    Int x Int -> Int
zero:     Unit -> Int
one:      Unit -> Int
plus(3, 4) = 7
times(3,4) = 12


In [24]:
def check_commutative(op, name):
    # op ∘ symmetry == op
    swapped = compose(symmetry(INT_PAIR), op)

    for a, b in [(2, 3), (0, 5), (-2, 7)]:
        lhs = run_int_from_pair(op, (a, b))
        rhs = run_int_from_pair(swapped, (a, b))
        print(f"{name} comm ({a}, {b}):", lhs, "==", rhs, lhs == rhs)


def check_associative(op, name):
    # left:  ((A × A) × A) --(op × id)--> A × A --op--> A
    left = compose(par(op, identity(INT)), op)

    # right: ((A × A) × A) --assoc--> A × (A × A) --(id × op)--> A × A --op--> A
    right = compose(
        compose(assoc(INT_TRIPLE_L), par(identity(INT), op)),
        op,
    )

    for a, b, c in [(1, 2, 3), (0, 4, 5), (-2, 7, 3)]:
        arg = nested_left_arg(a, b, c)
        lhs = int_val(run(left, arg, ctx, graph))
        rhs = int_val(run(right, arg, ctx, graph))
        print(f"{name} assoc ({a}, {b}, {c}):", lhs, "==", rhs, lhs == rhs)

In [25]:
check_commutative(int_semiring.plus, "plus")
check_associative(int_semiring.plus, "plus")

check_commutative(int_semiring.times, "times")
check_associative(int_semiring.times, "times")

plus comm (2, 3): 5 == 5 True
plus comm (0, 5): 5 == 5 True
plus comm (-2, 7): 5 == 5 True
plus assoc (1, 2, 3): 6 == 6 True
plus assoc (0, 4, 5): 9 == 9 True
plus assoc (-2, 7, 3): 8 == 8 True
times comm (2, 3): 6 == 6 True
times comm (0, 5): 0 == 0 True
times comm (-2, 7): -14 == -14 True
times assoc (1, 2, 3): 6 == 6 True
times assoc (0, 4, 5): 0 == 0 True
times assoc (-2, 7, 3): -42 == -42 True


## 14. Recursive Schemes — cata, ana, hylo

Catamorphisms (`cata`), anamorphisms (`ana`), and hylomorphisms (`hylo`) are uniform recursion patterns over an `Optic` fixed point.

An `Optic` becomes a recursive carrier by supplying `forward = unroll` and `backward = roll` together with an explicit `carrier` type.

The demo below uses a trivially terminating carrier over `Int` with shape `F(X) = 1 + X`:
- `cata` folds: the algebra `1 + Int → Int` always returns 7
- `ana` unfolds: the coalgebra `Int → 1 + Int` always stops immediately, rolling to 42
- `hylo` chains them

In [26]:
fp = one_or_self_optic(rolled_value=42)

stop_coalg = compose(delete(INT), inl(SumType(UNIT, INT)))
fold_alg   = case(const_int(7), identity(INT))

folded    = cata(fp, fold_alg)
unfolded  = ana(fp, stop_coalg)
transform = hylo(fp, stop_coalg, fold_alg)

print("cata: algebra folds any input to 7")
print("  cata(999) =", run_int(folded, 999))

print("ana:  coalgebra stops immediately, rolls to 42")
print("  ana(5) =", run_int(unfolded, 5))

print("hylo: unfold then fold → always 7")
print("  hylo(999) =", run_int(transform, 999))

cata: algebra folds any input to 7
  cata(999) = 7
ana:  coalgebra stops immediately, rolls to 42
  ana(5) = 42
hylo: unfold then fold → always 7
  hylo(999) = 7


## 15. Backend Morphisms

`BackendOps.from_spec` loads a JSON spec and registers each operation as a Hydra primitive with a canonical name (`unialg.backend.<op>`).  The same name is used regardless of which library backs it — swapping backends means changing the spec file, not the DSL.

`run` automatically includes `aux_primitives` from the morphism, so no manual graph extension is needed.

In [27]:
import math as _math
import numpy as np
from unialg import compile_program

def run_f1(op, a):
    prog = compile_program(f"load numpy\nlet f = {op}")
    return prog.run(np.array([a]))[0]

def run_f2(op, a, b):
    prog = compile_program(f"load numpy\nlet f = {op}")
    return prog.run(np.array([a]), np.array([b]))[0]

# --- elementwise binary ops ---
print("add(2, 3)          =", run_f2("add",       2.0,  3.0))
print("subtract(5, 1.5)   =", run_f2("subtract",  5.0,  1.5))
print("multiply(2, 3)     =", run_f2("multiply",  2.0,  3.0))
print("divide(9, 2)       =", run_f2("divide",    9.0,  2.0))
print("power(2, 10)       =", run_f2("power",     2.0, 10.0))
print("maximum(3, 7)      =", run_f2("maximum",   3.0,  7.0))
print("minimum(3, 7)      =", run_f2("minimum",   3.0,  7.0))
print("logaddexp(0, 0)    =", run_f2("logaddexp", 0.0,  0.0))
print()

# --- unary ops ---
print("exp(1)             =", run_f1("exp",       1.0))
print("log(e)             =", run_f1("log",   _math.e))
print("log1p(0)           =", run_f1("log1p",     0.0))
print("sqrt(16)           =", run_f1("sqrt",     16.0))
print("square(5)          =", run_f1("square",    5.0))
print("abs(-3.5)          =", run_f1("abs",      -3.5))
print("neg(4)             =", run_f1("neg",       4.0))
print("tanh(0)            =", run_f1("tanh",      0.0))
print("reciprocal(4)      =", run_f1("reciprocal", 4.0))
print("sigmoid(0)         =", run_f1("sigmoid",   0.0))


add(2, 3)          = 5.0
subtract(5, 1.5)   = 3.5
multiply(2, 3)     = 6.0
divide(9, 2)       = 4.5
power(2, 10)       = 1024.0
maximum(3, 7)      = 7.0
minimum(3, 7)      = 3.0
logaddexp(0, 0)    = 0.6931471805599453

exp(1)             = 2.718281828459045
log(e)             = 1.0
log1p(0)           = 0.0
sqrt(16)           = 4.0
square(5)          = 25.0
abs(-3.5)          = 3.5
neg(4)             = -4.0
tanh(0)            = 0.0
reciprocal(4)      = 0.25
sigmoid(0)         = 0.5


## 16. Semiring Factory

Build typed semirings from backend op names. The factory resolves both the elementwise op and its reduction form automatically — you only name the base op.

In [28]:
# Semiring factory removed (tensor module deferred).
# See greenfield branch for the original semiring_factory demo.


In [29]:
# Semiring factory deferred — see greenfield branch.


In [30]:
# Semiring factory deferred — see greenfield branch.


In [31]:
# Semiring factory deferred — see greenfield branch.


### Custom / learned semiring ops

Register any callable as a semiring op. Provide `reduce_fn` to make it usable as a fold in contractions.

In [32]:
# Semiring factory deferred — see greenfield branch.


In [33]:
# Notebook cell: use the current unialg API without rewriting library code.

import hydra.lexical as L
import hydra.sources.libraries as Libs
import hydra.dsl.meta.phantoms as P

from hydra.core import (
    IntegerType,
    LiteralTypeInteger,
    Name,
    TypeLiteral,
)

from unialg.syntax import expressions as expr
from unialg.semantics import functors as F
from unialg.semantics import morphisms as M
from unialg.main import run


# Match the existing notebook/test pattern: build a Hydra graph with registered stdlib primitives.
def make_graph():
    primitives = []
    for attr in dir(Libs):
        if attr.startswith("register_") and attr.endswith("_primitives"):
            primitives.extend(getattr(Libs, attr)().values())
    return L.graph_with_primitives(primitives, ())


ctx = L.empty_context()
graph = make_graph()


# Basic Hydra Int type used by the examples.
INT = TypeLiteral(LiteralTypeInteger(IntegerType.INT32))

ADD = Name("hydra.lib.math.add")
MUL = Name("hydra.lib.math.mul")


# Existing boundary pattern:
# raw Hydra lambda -> expr.Prim(raw, dom, cod) -> M.Morphism(...)
def unary_int_prim(op: Name, rhs: int) -> M.Morphism:
    x = P.var("x")
    raw = P.lam("x", P.primitive2(op, x, P.int32(rhs))).value
    return M.Morphism(expr.Prim(raw, INT, INT))


add1 = unary_int_prim(ADD, 1)
double = unary_int_prim(MUL, 2)


# A polynomial functor shape using your existing constructors:
# F(X) = Maybe[X] × List[X]
shape = F.Functor(
    "MaybeAndList",
    F.prod(
        F.maybe(F.id_()),
        F.list_(F.id_()),
    ),
)

print("F(X) =", expr.pretty(shape.body))
print("F(INT) =", shape.apply(INT))


# Arrow action:
# add1 : INT -> INT
# shape.map(add1) : F(INT) -> F(INT)
lifted_add1 = shape.map(add1)

print("F(add1) domain   =", lifted_add1.dom())
print("F(add1) codomain =", lifted_add1.cod())
print("F(add1) syntax   =", expr.pretty(lifted_add1.node))


# Functor composition:
# outer(X) = X × X
# inner(X) = Maybe[List[X]]
# outer.compose(inner) gives outer ∘ inner.
outer = F.Functor("Pair", F.prod(F.id_(), F.id_()))
inner = F.Functor("MaybeList", F.maybe(F.list_(F.id_())))

composed = outer.compose(inner)

print("(outer ∘ inner)(X) =", expr.pretty(composed.body))
print(
    "compose/apply law holds on INT =",
    composed.apply(INT) == outer.apply(inner.apply(INT)),
)

# Runtime smoke check through your existing lowering boundary.
def int_value(term) -> int:
    return term.value.value.value


print("add1(5)   =", int_value(run(add1, P.int32(5).value, ctx, graph)))
print("double(5) =", int_value(run(double, P.int32(5).value, ctx, graph)))

F(X) = Maybe[X] * List[X]
F(INT) = TypePair(value=PairType(first=TypeMaybe(value=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>))), second=TypeList(value=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>)))))
F(add1) domain   = TypePair(value=PairType(first=TypeMaybe(value=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>))), second=TypeList(value=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>)))))
F(add1) codomain = TypePair(value=PairType(first=TypeMaybe(value=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>))), second=TypeList(value=TypeLiteral(value=LiteralTypeInteger(value=<IntegerType.INT32: Name(value='int32')>)))))
F(add1) syntax   = F(Maybe[X] * List[X])
(outer ∘ inner)(X) = Maybe[List[X]] * Maybe[List[X]]
compose/apply law holds on INT = True
add1(5)   = 6
double(5) = 10


In [34]:
# Register ONLY the five NumPy atoms:
#
#   1. einsum
#   2. add
#   3. softmax
#   4. gelu
#   5. layer_norm
#
# Everything else should be built later using your Morphism / Functor machinery.

import hashlib
import re
import numpy as np

import hydra.dsl.terms as Terms
import hydra.dsl.meta.phantoms as P

from hydra.core import (
    Name,
    TypeScheme,
    TypeLiteral,
    LiteralTypeFloat,
    FloatType,
    TermList,
    TermPair,
    TermLiteral,
    TermUnit,
)
from hydra.graph import Primitive
from hydra.dsl.python import Right, Nothing

from unialg.objects import ExpType, ProductType, ListType
from unialg.syntax import expressions as expr
from unialg.semantics import morphisms as M


FLOAT64 = TypeLiteral(LiteralTypeFloat(FloatType.FLOAT64))


# ---------------------------------------------------------------------
# Minimal tensor codec for notebook tests
# ---------------------------------------------------------------------

def term_to_py(term):
    match term:
        case TermList(value=xs):
            return [term_to_py(x) for x in xs]

        case TermPair(value=p):
            left, right = p
            return (term_to_py(left), term_to_py(right))

        case TermLiteral(value=lit):
            v = lit
            while hasattr(v, "value"):
                v = v.value
            return v

        case TermUnit():
            return None

        case _:
            raise TypeError(f"term_to_py: unsupported {type(term).__name__}: {term!r}")


def py_to_term(value):
    if isinstance(value, np.ndarray):
        return py_to_term(value.tolist())

    if isinstance(value, list):
        return Terms.list_([py_to_term(x) for x in value])

    if isinstance(value, tuple) and len(value) == 2:
        return Terms.pair(py_to_term(value[0]), py_to_term(value[1]))

    if isinstance(value, (float, np.floating, int, np.integer)):
        return Terms.float64(float(value))

    raise TypeError(f"py_to_term: unsupported {type(value).__name__}: {value!r}")


def as_array(term):
    return np.asarray(term_to_py(term), dtype=np.float64)


# ---------------------------------------------------------------------
# Existing primitive registration pattern
# ---------------------------------------------------------------------

def _safe_name(text: str) -> str:
    clean = re.sub(r"[^a-zA-Z0-9_]+", "_", text).strip("_")
    digest = hashlib.sha1(text.encode()).hexdigest()[:8]
    return f"{clean}_{digest}" if clean else digest


def numpy_atom(name: str, dom, cod, fn) -> M.Morphism:
    """
    Register one NumPy-backed atom as a Hydra primitive and wrap it
    as a unialg Morphism.

    This is the only primitive-registration mechanism used here.
    """
    prim_name = Name(f"notebook.numpy_atoms.{name}")

    def impl(_ctx, _graph, args):
        if len(args) != 1:
            raise ValueError(f"{name}: expected one argument, got {len(args)}")

        out = fn(args[0])
        return Right(py_to_term(out))

    prim = Primitive(
        prim_name,
        TypeScheme((), ExpType(dom, cod), Nothing()),
        impl,
    )

    x = P.var("x")
    raw = P.lam("x", P.apply(P.primitive(prim_name), x)).value

    return M.Morphism(
        expr.Prim(raw, dom, cod),
        aux_primitives=(prim,),
    )


# ---------------------------------------------------------------------
# The five atom families
# ---------------------------------------------------------------------

def einsum(equation: str, dom, cod) -> M.Morphism:
    """
    Atom 1: NumPy einsum.

    The input term may be:
      - a single tensor
      - a pair of tensors
      - nested pairs, flattened left-to-right

    Example:
      einsum("td,dh->th", ProductType(SEQ, MATRIX), HEADSEQ)
    """
    def flatten_operands(x):
        if isinstance(x, tuple) and len(x) == 2:
            return [*flatten_operands(x[0]), *flatten_operands(x[1])]
        return [x]

    def fn(term):
        operands = [np.asarray(x, dtype=np.float64) for x in flatten_operands(term_to_py(term))]
        return np.einsum(equation, *operands)

    return numpy_atom(
        f"einsum_{_safe_name(equation)}",
        dom,
        cod,
        fn,
    )


def add(tensor_type) -> M.Morphism:
    """
    Atom 2: NumPy add.

    (X, X) -> X
    """
    dom = ProductType(tensor_type, tensor_type)

    def fn(term):
        left, right = term_to_py(term)
        return np.add(
            np.asarray(left, dtype=np.float64),
            np.asarray(right, dtype=np.float64),
        )

    return numpy_atom(
        "add",
        dom,
        tensor_type,
        fn,
    )


def softmax(tensor_type, axis: int = -1) -> M.Morphism:
    """
    Atom 3: softmax using NumPy exp/sum/max.

    X -> X
    """
    def fn(term):
        x = as_array(term)
        x = x - np.max(x, axis=axis, keepdims=True)
        e = np.exp(x)
        return e / np.sum(e, axis=axis, keepdims=True)

    return numpy_atom(
        f"softmax_axis_{axis}",
        tensor_type,
        tensor_type,
        fn,
    )


def gelu(tensor_type) -> M.Morphism:
    """
    Atom 4: GELU activation using NumPy tanh/sqrt.

    X -> X
    """
    def fn(term):
        x = as_array(term)
        return 0.5 * x * (
            1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3))
        )

    return numpy_atom(
        "gelu",
        tensor_type,
        tensor_type,
        fn,
    )


def layer_norm(tensor_type, axis: int = -1, eps: float = 1e-5) -> M.Morphism:
    """
    Atom 5: layer normalization using NumPy mean/var/sqrt.

    X -> X
    """
    def fn(term):
        x = as_array(term)
        mean = np.mean(x, axis=axis, keepdims=True)
        var = np.var(x, axis=axis, keepdims=True)
        return (x - mean) / np.sqrt(var + eps)

    return numpy_atom(
        f"layer_norm_axis_{axis}",
        tensor_type,
        tensor_type,
        fn,
    )


# ---------------------------------------------------------------------
# Export atom constructors in one place
# ---------------------------------------------------------------------

atoms = {
    "einsum": einsum,
    "add": add,
    "softmax": softmax,
    "gelu": gelu,
    "layer_norm": layer_norm,
}

print("Registered NumPy atom constructors:")
print("  atoms['einsum'](equation, dom, cod)")
print("  atoms['add'](tensor_type)")
print("  atoms['softmax'](tensor_type, axis=-1)")
print("  atoms['gelu'](tensor_type)")
print("  atoms['layer_norm'](tensor_type, axis=-1, eps=1e-5)")

Registered NumPy atom constructors:
  atoms['einsum'](equation, dom, cod)
  atoms['add'](tensor_type)
  atoms['softmax'](tensor_type, axis=-1)
  atoms['gelu'](tensor_type)
  atoms['layer_norm'](tensor_type, axis=-1, eps=1e-5)


In [35]:
# SECOND CELL ONLY:
# Build a transformer from existing unialg morphisms/functors and the five atoms.
#
# Assumes previous cell already defined:
#   atoms
#   py_to_term
#   term_to_py
#   FLOAT64
#   ctx
#   graph
#
# This cell does NOT:
#   - use expr.Prim
#   - register new primitives
#   - define layer primitives
#   - wrap compose/pair/identity
#   - create a custom Morphism type

import numpy as np

from unialg.objects import ListType, ProductType, show_type
from unialg.semantics.morphisms import Morphism, compose, pair, identity
from unialg.semantics.functors import Functor, list_, id_, prod
from unialg.main import run


missing = [
    name for name in ("atoms", "py_to_term", "term_to_py", "FLOAT64", "ctx", "graph")
    if name not in globals()
]
if missing:
    raise NameError(f"Missing required notebook globals from earlier cells: {missing}")


# ---------------------------------------------------------------------
# Atom constructors from the previous cell
# ---------------------------------------------------------------------

einsum = atoms["einsum"]
add = atoms["add"]
softmax = atoms["softmax"]
gelu = atoms["gelu"]
layer_norm = atoms["layer_norm"]


# ---------------------------------------------------------------------
# Object types
# ---------------------------------------------------------------------

D_MODEL = 4
D_HEAD = 4
D_FF = 8
T_SEQ = 5

SCALAR = FLOAT64

VEC = ListType(SCALAR)          # list<float64>
SEQ = ListType(VEC)             # list<list<float64>>
MAT = ListType(ListType(SCALAR))
SCORES = MAT

print("VEC    =", show_type(VEC))
print("SEQ    =", show_type(SEQ))
print("MAT    =", show_type(MAT))
print("SCORES =", show_type(SCORES))


# ---------------------------------------------------------------------
# Derived morphism builders
# ---------------------------------------------------------------------
# These do not register primitives and do not construct Prim nodes.
# They only reuse atom morphism nodes and expose them through the repo's
# existing Morphism(param=...) mechanism.
#
# Because Morphism(param=P) expects the raw domain to be P × A,
# the einsum equation operand order below is:
#
#   parameter first, input second
#
# Example:
#   W : d×h
#   x : t×d
#   equation = "dh,td->th"


def linear(weight_type, input_type, output_type, equation):
    op = einsum(
        equation,
        ProductType(weight_type, input_type),
        output_type,
    )

    return Morphism(
        node=op.node,
        param=weight_type,
        monad=op.monad,
        aux_primitives=op.aux_primitives,
    )


def residual(norm, branch, space):
    # x -> (x, branch(x)) -> x + branch(x) -> norm(...)
    return compose(
        compose(
            pair(identity(space), branch),
            add(space),
        ),
        norm,
    )


# ---------------------------------------------------------------------
# Atomic morphism instances
# ---------------------------------------------------------------------
# Still only the five atom families:
#   einsum / add / softmax / gelu / layer_norm

add_seq = add(SEQ)
softmax_scores = softmax(SCORES, axis=-1)
gelu_vec = gelu(VEC)
norm_seq = layer_norm(SEQ, axis=-1)

score_qk = einsum(
    "th,sh->ts",
    ProductType(SEQ, SEQ),
    SCORES,
)

mix_wv = einsum(
    "ts,sh->th",
    ProductType(SCORES, SEQ),
    SEQ,
)


# ---------------------------------------------------------------------
# Parametric linear projections from einsum atoms
# ---------------------------------------------------------------------
# These are not primitives. Each one is:
#
#   Morphism(node=<einsum atom node>, param=MAT)
#
# so the visible arrow hides the parameter prefix.

q = linear(MAT, SEQ, SEQ, "dh,td->th")
k = linear(MAT, SEQ, SEQ, "dh,td->th")
v = linear(MAT, SEQ, SEQ, "dh,td->th")
o = linear(MAT, SEQ, SEQ, "hd,th->td")

ffn_1 = linear(MAT, VEC, VEC, "df,d->f")
ffn_2 = linear(MAT, VEC, VEC, "fd,f->d")


# ---------------------------------------------------------------------
# Attention assembled from existing morphism combinators
# ---------------------------------------------------------------------

# x -> (q(x), k(x))
qk = pair(q, k)

# x -> q(x) @ k(x)^T
scores = compose(qk, score_qk)

# x -> softmax(scores(x))
weights = compose(scores, softmax_scores)

# x -> (weights(x), v(x))
weights_and_values = pair(weights, v)

# x -> weights(x) @ v(x)
attn_head = compose(weights_and_values, mix_wv)

# x -> output projection
attention = compose(attn_head, o)


# ---------------------------------------------------------------------
# FFN assembled from existing morphism combinators
# ---------------------------------------------------------------------

token_ffn = compose(
    compose(ffn_1, gelu_vec),
    ffn_2,
)


# ---------------------------------------------------------------------
# Lift token FFN over sequence using existing polynomial functor machinery
# ---------------------------------------------------------------------

Tokenwise = Functor("Tokenwise", list_(id_()))
ffn_seq = Tokenwise.map(token_ffn)


# ---------------------------------------------------------------------
# Transformer blocks
# ---------------------------------------------------------------------

attn_block = residual(norm_seq, attention, SEQ)
ffn_block = residual(norm_seq, ffn_seq, SEQ)

transformer = compose(attn_block, ffn_block)


# ---------------------------------------------------------------------
# Inspect constructed program
# ---------------------------------------------------------------------

print("\nConstructed morphisms:")
for name, morph in [
    ("q", q),
    ("k", k),
    ("v", v),
    ("o", o),
    ("qk", qk),
    ("scores", scores),
    ("weights", weights),
    ("attention", attention),
    ("ffn_1", ffn_1),
    ("ffn_2", ffn_2),
    ("token_ffn", token_ffn),
    ("ffn_seq", ffn_seq),
    ("attn_block", attn_block),
    ("ffn_block", ffn_block),
    ("transformer", transformer),
]:
    print(f"  {name}: {show_type(morph.dom())} -> {show_type(morph.cod())}")
    print(f"      param: {show_type(morph.param)}")

print("\nFunctor:")
print("  Tokenwise(X) =", Tokenwise.body)
print("  ffn_seq node =", ffn_seq.node)

print("\nTransformer node:")
print(transformer.node)


# ---------------------------------------------------------------------
# Concrete parameter values
# ---------------------------------------------------------------------
# Scaling 1/sqrt(D_HEAD) is folded into Wk so no extra primitive is needed.

rng = np.random.default_rng(7)
scale = 0.35

Wq = rng.normal(0.0, scale, size=(D_MODEL, D_HEAD))
Wk = rng.normal(0.0, scale / np.sqrt(D_HEAD), size=(D_MODEL, D_HEAD))
Wv = rng.normal(0.0, scale, size=(D_MODEL, D_HEAD))
Wo = rng.normal(0.0, scale, size=(D_HEAD, D_MODEL))

W1 = rng.normal(0.0, scale, size=(D_MODEL, D_FF))
W2 = rng.normal(0.0, scale, size=(D_FF, D_MODEL))

x0 = rng.normal(0.0, 0.5, size=(T_SEQ, D_MODEL))


# ---------------------------------------------------------------------
# Parameter tree
# ---------------------------------------------------------------------
# Repo parameter composition order:
#
#   compose(f, g):  g.param × f.param
#   pair(f, g):     g.param × f.param
#
# So:
#
#   qk.param                 = Wk × Wq
#   scores.param             = Wk × Wq
#   weights.param            = Wk × Wq
#   weights_and_values.param = Wv × (Wk × Wq)
#   attention.param          = Wo × (Wv × (Wk × Wq))
#
#   token_ffn.param          = W2 × W1
#   ffn_seq.param            = W2 × W1
#
#   transformer.param        = ffn_block.param × attn_block.param
#                            = (W2 × W1) × (Wo × (Wv × (Wk × Wq)))

attn_params = (
    Wo,
    (
        Wv,
        (
            Wk,
            Wq,
        ),
    ),
)

ffn_params = (
    W2,
    W1,
)

transformer_params = (
    ffn_params,
    attn_params,
)

raw_arg = py_to_term(
    (
        transformer_params,
        x0,
    )
)


# ---------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------

y_term = run(transformer, raw_arg, ctx, graph)
y = np.asarray(term_to_py(y_term), dtype=np.float64)

print("\nInput shape: ", x0.shape)
print("Output shape:", y.shape)

print("\nInput:")
print(np.round(x0, 4))

print("\nOutput:")
print(np.round(y, 4))

print("\nPer-token output mean:")
print(np.round(y.mean(axis=-1), 8))

print("\nPer-token output std:")
print(np.round(y.std(axis=-1), 8))

VEC    = list<float64>
SEQ    = list<list<float64>>
MAT    = list<list<float64>>
SCORES = list<list<float64>>

Constructed morphisms:
  q: list<list<float64>> -> list<list<float64>>
      param: list<list<float64>>
  k: list<list<float64>> -> list<list<float64>>
      param: list<list<float64>>
  v: list<list<float64>> -> list<list<float64>>
      param: list<list<float64>>
  o: list<list<float64>> -> list<list<float64>>
      param: list<list<float64>>
  qk: list<list<float64>> -> (list<list<float64>>, list<list<float64>>)
      param: (list<list<float64>>, list<list<float64>>)
  scores: list<list<float64>> -> list<list<float64>>
      param: (list<list<float64>>, list<list<float64>>)
  weights: list<list<float64>> -> list<list<float64>>
      param: (list<list<float64>>, list<list<float64>>)
  attention: list<list<float64>> -> list<list<float64>>
      param: (list<list<float64>>, (list<list<float64>>, (list<list<float64>>, list<list<float64>>)))
  ffn_1: list<float64> -> list<float6

## 17. Surface Syntax — load and let

The `syntax/` layer provides a Pratt parser that turns ASCII expressions into `MorphismExpr` trees.
A `load <backend>` directive binds every operation from the backend spec into the parser's environment,
so subsequent `let` definitions resolve to real backend `Prim` nodes rather than unresolved `Ref` placeholders.

Morphism operators: `>>` compose, `&` pair, `||` parallel, `|` case, `^n` copy, `[0]`/`[1]` project, `?0`/`?1` inject.
Functor operators: `*` list, `&` product, `|` sum.
Functor mapping: `x*{f}` means `List.map(f)`.

In [36]:
from unialg import compile_program

src = """
shape Nat = 1 | x

let zero = ! >> |0
let successor = |1

let one = zero >> successor
let two = one >> Nat{successor}
let three = two >> Nat{Nat{successor}}

let count = three
"""

compiled = compile_program(src)
print(compiled.run())

('right', ('right', ('right', ('left', None))))


## Focus syntax smoke test: pure DSL recursion surface

This checks the explicit `focus` plus `cata`/`ana`/`hylo` syntax without loading a backend. 
The following cell keeps using ordinary integer morphisms as semantic inputs; the next cell uses the recursive carrier syntax for Peano-style natural numbers.


In [37]:
from unialg.syntax.parse import parse_program
from unialg.semantics.construct import construct_program

# This is deliberately backend-free. It uses the pure integer morphisms
# already defined above in the notebook as semantic inputs to the DSL.
focus_src = """
shape Id = x

shape self : Id <-> Id by add1 / double

let folded = cata[self](add1)
let built = ana[self](add1)
let transformed = hylo[self](add1, add1)
"""

program = parse_program(focus_src)
constructed = construct_program(program, {"add1": add1, "double": double})

print("focuses:", list(constructed.focuses))
for name in ("folded", "built", "transformed"):
    print(f"{name}:", show_morphism(constructed.morphisms[name]))

print("\nCarrier syntax is available below; Peano addition can now be written as a fold.")


focuses: ['self']
folded: Int -> Int
built: Int -> Int
transformed: Int -> Int

Carrier syntax is available below; Peano addition can now be written as a fold.


## Peano addition and multiplication in pure DSL

This uses no backend. `Nat` is declared as the fixed point of `NatF = 1 | x`; 
`zero` and `succ` are built from `roll[nat]`; addition and multiplication are folds.


In [45]:
from unialg import compile_program

peano_src = """
shape NatF = 1 | x
shape Nat = fix NatF

let zero = |0 >> roll[Nat]
let succ = |1 >> roll[Nat]

let one = zero >> succ
let two = one >> succ
let three = two >> succ

# add(n) : Nat -> Nat
# add(n)(zero)   = n
# add(n)(succ k) = succ(add(n)(k))
let add(n) = cata[Nat](n | succ)

# mul(n) : Nat -> Nat
# mul(n)(zero)   = zero
# mul(n)(succ k) = add(n)(mul(n)(k))
let mul(n) = cata[Nat](zero | add(n))

let five = three >> add(two)
let six = three >> mul(two)
"""

def peano_to_int(value):
    n = 0
    while True:
        tag, payload = value
        if tag == "left":
            return n
        if tag != "right":
            raise ValueError(f"unexpected Peano layer: {value!r}")
        n += 1
        value = payload

five = compile_program(peano_src, target="five").run()
six = compile_program(peano_src, target="six").run()

print("3 + 2 =", peano_to_int(five), five)
print("3 * 2 =", peano_to_int(six), six)

assert peano_to_int(five) == 5
assert peano_to_int(six) == 6


3 + 2 = 5 ('right', ('right', ('right', ('right', ('right', ('left', None))))))
3 * 2 = 6 ('right', ('right', ('right', ('right', ('right', ('right', ('left', None)))))))


## Anamorphism and hylomorphism smoke test

Here `unroll[nat]` is used as a coalgebra: it exposes one Peano layer. 
`ana[nat](unroll[nat])` rebuilds a `Nat` from those exposed layers, while `hylo[nat](unroll[nat], zero | succ)` unfolds and immediately folds back to the same value.


In [39]:
from unialg import compile_program

ana_hylo_src = """
shape NatF = 1 | x
shape Nat = fix NatF

let zero = |0 >> roll[Nat]
let succ = |1 >> roll[Nat]

let one = zero >> succ
let two = one >> succ
let three = two >> succ

# unroll[Nat] : Nat -> NatF Nat, so it is a coalgebra for ana.
let rebuild = ana[Nat](unroll[Nat])

# zero | succ : NatF Nat -> Nat, so it is the matching algebra for cata.
let fold_id = cata[Nat](zero | succ)

# hylo unfolds with the coalgebra, then folds with the algebra.
let hylo_id = hylo[Nat](unroll[Nat], zero | succ)

let rebuilt_three = three >> rebuild
let folded_three = three >> fold_id
let hylo_three = three >> hylo_id
"""

def peano_to_int(value):
    n = 0
    while True:
        tag, payload = value
        if tag == "left":
            return n
        if tag != "right":
            raise ValueError(f"unexpected Peano layer: {value!r}")
        n += 1
        value = payload

for let in ("rebuilt_three", "folded_three", "hylo_three"):
    value = compile_program(ana_hylo_src, target=name).run()
    print(f"{name} =", peano_to_int(value), value)
    assert peano_to_int(value) == 3


KeyError: "compile_program: unknown target 'transformed'"

In [ ]:
import time
import numpy as np
from unialg import compile_program

dot = compile_program("""
load numpy
let dot = multiply >> reduce.add
""")

# --- correctness check ---
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
assert np.isclose(dot.run(a, b), np.dot(a, b))

# --- benchmark ---
rng = np.random.default_rng(0)
m, k, n = 64, 128, 64

A = rng.standard_normal((m, k))
B = rng.standard_normal((k, n))

# warm-up
_ = np.array([[dot.run(A[i], B[:, j]) for j in range(n)] for i in range(m)])

REPS = 5

t0 = time.perf_counter()
for _ in range(REPS):
    C = np.array([[dot.run(A[i], B[:, j]) for j in range(n)] for i in range(m)])
elapsed_let = (time.perf_counter() - t0) / REPS

t0 = time.perf_counter()
for _ in range(REPS):
    C_ref = A @ B
elapsed_numpy = (time.perf_counter() - t0) / REPS

assert np.allclose(C, C_ref)

print(f"Shape: A{A.shape} @ B{B.shape} -> C{C.shape}")
print(f"  let (dot loop): {elapsed_let*1e3:.2f} ms/call")
print(f"  numpy matmul:     {elapsed_numpy*1e6:.2f} µs/call")
print(f"  overhead factor:  {elapsed_let/elapsed_numpy:.0f}x")
print(f"  result correct:   {np.allclose(C, C_ref)}")


Shape: A(64, 128) @ B(128, 64) -> C(64, 64)
  let (dot loop): 2090.37 ms/call
  numpy matmul:     72.62 µs/call
  overhead factor:  28786x
  result correct:   True


In [ ]:
from unialg import compile_program

src = """
shape FoldKind = 1 | 1 | 1 | 1
shape SeqF = 1 | FoldKind & x
shape FoldSeq = fix SeqF

let valley = |0 >> |0 >> |0
let petal  = |0 >> |1

let nil  = |0 >> roll[FoldSeq]
let cons = |1 >> roll[FoldSeq]

let two_folds = (valley & ((petal & nil) >> cons)) >> cons
"""

p = compile_program(src)
print(p.input_type, "->", p.output_type)

def show(v, depth=0):
    indent = "  " * depth
    if v is None:
        return f"{indent}unit"
    if isinstance(v, tuple) and len(v) == 2:
        tag, inner = v
        return f"{indent}{tag}:\n{show(inner, depth+1)}"
    if isinstance(v, tuple):
        return f"{indent}pair:\n" + "\n".join(show(x, depth+1) for x in v)
    return f"{indent}{v!r}"

print(show(p.run()))

TypeVariable(value=Name(value='t0')) -> TypeVariable(value=Name(value='unialg.carrier.FoldSeq'))
right:
  ('left', ('left', ('left', None))):
    right:
      ('right', ('left', None)):
        left:
          unit


In [ ]:
# Tensor-extension transformer/encoder demo with functor action.
#
# Legacy reference: /home/scanbot/unified-algebra/examples/transformer.ua
#
# This cell demonstrates the intended interaction:
#   - tensor contractions are ordinary morphisms
#   - functors act on those morphisms too
#   - attention and FFN residual wiring are composed in DSL syntax
#   - PyTorch is only the reference oracle
#
# Current DSL/backend limits shown honestly here:
#   - torch softmax currently uses PyTorch's implicit dim behavior
#   - relu and layer_norm are not currently registered axis-aware DSL morphisms
#   - tanh stands in for the FFN activation until activation coverage is expanded
#   - composing attention + FFN into one morphism still needs better routing of
#     carried parameter values through intermediate stages
#
# Temporary notebook workaround: larger DSL strings can currently drive the
# parser-combinator stack deep enough to need a higher recursion limit. The
# underlying lexer/parser robustness issue is recorded in NEXT_STEPS_TENSOR.md.

import sys
sys.setrecursionlimit(20000)

import time

import numpy as np
import torch
from unialg import compile_program

transformer_src = """
load torch
algebra real(plus=add, times=multiply, zero=0.0, one=1.0)
shape Batch = List[x]

let project = contract[real]("td,dh->th")
let project_batch_probe = Batch{project}

let score = contract[real]("th,sh->ts")
let score_weights = score >> softmax
let mix = contract[real]("ts,sh->th")
let attention = (score_weights || id) >> mix
let out_proj = contract[real]("th,hd->td")
let attention_out = (attention || id) >> out_proj

let qkv = (project || project) || project
let attention_from_inputs = (qkv || id) >> attention_out
let attention_block = (attention_from_inputs || id) >> add
let batched_attention_block = Batch{attention_block}

let ffn_in = contract[real]("td,df->tf")
let ffn_out = contract[real]("tf,fd->td")
let ffn_core = ((ffn_in >> tanh) || id) >> ffn_out
let ffn_with_residual_input = id & ([0] >> [0])
let ffn_block = ffn_with_residual_input >> (ffn_core || id) >> add
let batched_ffn_block = Batch{ffn_block}
"""

project = compile_program(transformer_src, target="project")
project_batch_probe = compile_program(transformer_src, target="project_batch_probe")
attention_block = compile_program(transformer_src, target="attention_block")
batched_attention_block = compile_program(transformer_src, target="batched_attention_block")
ffn_block = compile_program(transformer_src, target="ffn_block")
batched_ffn_block = compile_program(transformer_src, target="batched_ffn_block")


def softmax_reference(x):
    # Matches torch.nn.functional.softmax's current implicit dim behavior for this 2D input.
    # This warning-producing default is a backend-spec gap; axis-aware softmax is in next steps.
    return torch.nn.functional.softmax(x, dim=-1)


def attention_reference(x, Wq, Wk, Wv, Wo):
    q = x @ Wq
    k = x @ Wk
    v = x @ Wv
    weights = softmax_reference(q @ k.T)
    return x + (weights @ v) @ Wo


def ffn_reference(x, W1, W2):
    return x + torch.tanh(x @ W1) @ W2


def encoder_reference(x, Wq, Wk, Wv, Wo, W1, W2):
    return ffn_reference(attention_reference(x, Wq, Wk, Wv, Wo), W1, W2)


rng = np.random.default_rng(7)
t, d_model, d_head, d_ff = 4, 6, 3, 8

x1 = torch.tensor(rng.standard_normal((t, d_model)), dtype=torch.float64)
x2 = torch.tensor(rng.standard_normal((t, d_model)), dtype=torch.float64)
Wq = torch.tensor(rng.standard_normal((d_model, d_head)), dtype=torch.float64)
Wk = torch.tensor(rng.standard_normal((d_model, d_head)), dtype=torch.float64)
Wv = torch.tensor(rng.standard_normal((d_model, d_head)), dtype=torch.float64)
Wo = torch.tensor(rng.standard_normal((d_head, d_model)), dtype=torch.float64)
W1 = torch.tensor(rng.standard_normal((d_model, d_ff)), dtype=torch.float64)
W2 = torch.tensor(rng.standard_normal((d_ff, d_model)), dtype=torch.float64)

# A contraction morphism works directly.
projected = project.run(x1, Wq)
assert torch.allclose(projected, x1 @ Wq)

# The same contraction morphism also works under functor action.
batched_projected = project_batch_probe.run([(x1, Wq), (x2, Wq)])
assert torch.allclose(batched_projected[0], x1 @ Wq)
assert torch.allclose(batched_projected[1], x2 @ Wq)

# Attention block: Q/K/V projection, score, softmax, value mixing, output projection, residual.
attention_arg1 = (((((x1, Wq), (x1, Wk)), (x1, Wv)), Wo), x1)
attention_arg2 = (((((x2, Wq), (x2, Wk)), (x2, Wv)), Wo), x2)
attention_result1 = attention_block.run(attention_arg1)
attention_result2 = attention_block.run(attention_arg2)
assert torch.allclose(attention_result1, attention_reference(x1, Wq, Wk, Wv, Wo))
assert torch.allclose(attention_result2, attention_reference(x2, Wq, Wk, Wv, Wo))

batched_attention = batched_attention_block.run([attention_arg1, attention_arg2])
assert torch.allclose(batched_attention[0], attention_result1)
assert torch.allclose(batched_attention[1], attention_result2)

# FFN block: tensor contraction, activation, tensor contraction, residual.
ffn_arg1 = ((attention_result1, W1), W2)
ffn_arg2 = ((attention_result2, W1), W2)
ffn_result1 = ffn_block.run(ffn_arg1)
ffn_result2 = ffn_block.run(ffn_arg2)
assert torch.allclose(ffn_result1, ffn_reference(attention_result1, W1, W2))
assert torch.allclose(ffn_result2, ffn_reference(attention_result2, W1, W2))

batched_ffn = batched_ffn_block.run([ffn_arg1, ffn_arg2])
assert torch.allclose(batched_ffn[0], ffn_result1)
assert torch.allclose(batched_ffn[1], ffn_result2)

# End-to-end reference parity for the encoder calculation represented by the two DSL morphisms.
assert torch.allclose(ffn_result1, encoder_reference(x1, Wq, Wk, Wv, Wo, W1, W2))
assert torch.allclose(ffn_result2, encoder_reference(x2, Wq, Wk, Wv, Wo, W1, W2))
for got in (*batched_attention, *batched_ffn):
    assert torch.all(torch.isfinite(got))

def timed(label, reps, fn):
    # Warm up once so timing mostly reflects steady-state execution.
    fn()
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    elapsed = (time.perf_counter() - t0) / reps
    print(f"{label:<28} {elapsed * 1e6:9.2f} us/call")
    return elapsed


REPS = 200

def dsl_attention_once():
    return attention_block.run(attention_arg1)


def torch_attention_once():
    return attention_reference(x1, Wq, Wk, Wv, Wo)


def dsl_ffn_once():
    return ffn_block.run(ffn_arg1)


def torch_ffn_once():
    return ffn_reference(attention_result1, W1, W2)


def dsl_encoder_two_stage_once():
    y = attention_block.run(attention_arg1)
    return ffn_block.run((y, W1), W2)


def torch_encoder_once():
    return encoder_reference(x1, Wq, Wk, Wv, Wo, W1, W2)


print("project shape:", x1.shape, "x", Wq.shape, "->", projected.shape)
print("Batch{project} probe shapes:", [item.shape for item in batched_projected])
print("attention block shape:", x1.shape, "->", attention_result1.shape)
print("Batch{attention_block} shapes:", [item.shape for item in batched_attention])
print("ffn block shape:", attention_result1.shape, "->", ffn_result1.shape)
print("Batch{ffn_block} shapes:", [item.shape for item in batched_ffn])
print("encoder reference parity:", bool(torch.allclose(ffn_result1, encoder_reference(x1, Wq, Wk, Wv, Wo, W1, W2))))
print()
print(f"Timing over {REPS} reps on this tiny torch encoder smoke shape:")
dsl_attn_t = timed("DSL attention", REPS, dsl_attention_once)
torch_attn_t = timed("Torch attention", REPS, torch_attention_once)
dsl_ffn_t = timed("DSL FFN", REPS, dsl_ffn_once)
torch_ffn_t = timed("Torch FFN", REPS, torch_ffn_once)
dsl_enc_t = timed("DSL attention + FFN", REPS, dsl_encoder_two_stage_once)
torch_enc_t = timed("Torch attention + FFN", REPS, torch_encoder_once)
print(f"attention overhead: {dsl_attn_t / torch_attn_t:.1f}x")
print(f"FFN overhead:       {dsl_ffn_t / torch_ffn_t:.1f}x")
print(f"encoder overhead:   {dsl_enc_t / torch_enc_t:.1f}x")


/home/scanbot/unialg/src/unialg/runtime/backend.py:210: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  py_result = fn(*py_args)


project shape: torch.Size([4, 6]) x torch.Size([6, 3]) -> torch.Size([4, 3])
Batch{project} probe shapes: [torch.Size([4, 3]), torch.Size([4, 3])]
attention block shape: torch.Size([4, 6]) -> torch.Size([4, 6])
Batch{attention_block} shapes: [torch.Size([4, 6]), torch.Size([4, 6])]
ffn block shape: torch.Size([4, 6]) -> torch.Size([4, 6])
Batch{ffn_block} shapes: [torch.Size([4, 6]), torch.Size([4, 6])]
encoder reference parity: True

Timing over 200 reps on this tiny torch encoder smoke shape:
DSL attention                 11769.17 us/call
Torch attention                   9.81 us/call
DSL FFN                        4133.80 us/call
Torch FFN                         3.53 us/call
DSL attention + FFN           15882.47 us/call
Torch attention + FFN            14.78 us/call
attention overhead: 1200.3x
FFN overhead:       1172.1x
encoder overhead:   1074.8x
